In [19]:
from pathlib import Path
import re

from unstructured.partition.pdf import partition_pdf

from langchain_text_splitters import (
    MarkdownHeaderTextSplitter,
    RecursiveCharacterTextSplitter,
)

from langchain_core.documents import Document

from sentence_transformers import SentenceTransformer

from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from dotenv import load_dotenv

In [4]:
def extract_page(text):

    match = re.search(r"<!-- PAGE: (\d+) -->", text)

    if match:
        return int(match.group(1))

    return None

In [12]:
def process_pdf(pdf_path):

    print(f"Processing: {pdf_path.name}")

    # =========================
    # 1. Extraction PDF
    # =========================

    elements = partition_pdf(
        filename=str(pdf_path),
        strategy="auto",
    )

    # =========================
    # 2. Convertir en markdown
    # =========================

    markdown_text = ""

    current_page = None

    for el in elements:

        text = el.text.strip()

        if not text:
            continue

        page = getattr(el.metadata, "page_number", None)

        # changement de page
        if page != current_page:

            markdown_text += (
                f"\n\n<!-- PAGE: {page} -->\n\n"
            )

            current_page = page

        # titres
        if el.category == "Title":

            markdown_text += f"\n# {text}\n"

        else:

            markdown_text += f"\n{text}\n"

    # =========================
    # 3. Markdown splitter
    # =========================

    headers_to_split_on = [
        ("#", "header"),
    ]

    markdown_splitter = MarkdownHeaderTextSplitter(
        headers_to_split_on=headers_to_split_on
    )

    header_docs = markdown_splitter.split_text(
        markdown_text
    )

    # =========================
    # 4. Enrichissement docs
    # =========================

    enhanced_docs = []

    for doc in header_docs:

        header = doc.metadata.get(
            "header",
            "Sans titre"
        )

        page = extract_page(doc.page_content)

        # nettoyage page marker
        clean_content = re.sub(
            r"<!-- PAGE: \d+ -->",
            "",
            doc.page_content
        ).strip()

        # ajout du header dans le contenu
        final_content = f"""
Section: {header}

{clean_content}
"""

        enhanced_doc = Document(
            page_content=final_content,
            metadata={
                "header": header,
                "page": page,
                "source": pdf_path.name,
            }
        )

        enhanced_docs.append(enhanced_doc)

    # =========================
    # 5. Recursive splitter
    # =========================

    recursive_splitter = (
        RecursiveCharacterTextSplitter(
            chunk_size=500,
            chunk_overlap=100,
        )
    )

    final_docs = recursive_splitter.split_documents(
        enhanced_docs
    )

    return final_docs

In [13]:
data_folder = Path("../data")

all_docs = []

pdf_files = list(data_folder.glob("*.pdf"))

print(f"{len(pdf_files)} PDFs found")

7 PDFs found


In [14]:
for pdf_file in pdf_files:

    try:

        docs = process_pdf(pdf_file)

        all_docs.extend(docs)

        print(
            f"{pdf_file.name} -> "
            f"{len(docs)} chunks"
        )

    except Exception as e:

        print(
            f"Error with {pdf_file.name}: {e}"
        )

Processing: 123_HMM_AG002_Agenda (en moyen & long séjour)_EX.pdf


No languages specified, defaulting to English.


123_HMM_AG002_Agenda (en moyen & long séjour)_EX.pdf -> 21 chunks
Processing: doc_.pdf


No languages specified, defaulting to English.
No features in text.
No features in text.
No features in text.


doc_.pdf -> 53 chunks
Processing: Guide paramétrage restrictions de visibilité agenda (1).pdf


...

Paramétrage Etablissement FILIERIS.pdf -> 345 chunks
Processing: [GU] Calcul des besoins V12.pdf


No languages specified, defaulting to English.


[GU] Calcul des besoins V12.pdf -> 77 chunks
Processing: [GU] Cotation à partir de l'agenda - 1.2307.pdf


No languages specified, defaulting to English.


[GU] Cotation à partir de l'agenda - 1.2307.pdf -> 127 chunks


In [15]:
pdf_files

[WindowsPath('../data/123_HMM_AG002_Agenda (en moyen & long séjour)_EX.pdf'),
 WindowsPath('../data/doc_.pdf'),
 WindowsPath('../data/Guide paramétrage restrictions de visibilité agenda (1).pdf'),
 WindowsPath('../data/Guide Utilisateur HM Agenda v1.24.11.pdf'),
 WindowsPath('../data/Paramétrage Etablissement FILIERIS.pdf'),
 WindowsPath('../data/[GU] Calcul des besoins V12.pdf'),
 WindowsPath("../data/[GU] Cotation à partir de l'agenda - 1.2307.pdf")]

In [16]:
print(len(all_docs))

1902


In [17]:
all_docs

[Document(metadata={'header': 'Sans titre', 'page': 2, 'source': '123_HMM_AG002_Agenda (en moyen & long séjour)_EX.pdf'}, page_content='Section: Sans titre'),
 Document(metadata={'header': 'Sans titre', 'page': 2, 'source': '123_HMM_AG002_Agenda (en moyen & long séjour)_EX.pdf'}, page_content='Table des matières 1 INTRODUCTION ..........................................................................................................................................2  \n2 EXERCICE N°1 – Planning de ressource ......................................................................................................2  \n2 EXERCICE N°2 – Prise de RDV .....................................................................................................................3  \n1'),
 Document(metadata={'header': 'Sans titre', 'page': 2, 'source': '123_HMM_AG002_Agenda (en moyen & long séjour)_EX.pdf'}, page_content='1  \nCahier d’exercices – AG002 – Agenda (en moyen et long séjour)  \n  \nCe cahier d’exe

In [20]:
# Embedding wrapper for LangChain
embedding_model = HuggingFaceEmbeddings(
    model_name="BAAI/bge-m3"
)

Loading weights: 100%|██████████| 391/391 [00:00<?, ?it/s]


In [21]:
from langchain_chroma import Chroma

vectordb = Chroma.from_documents(
    documents=all_docs,
    embedding=embedding_model,
    persist_directory="./all_docs_embedding"
)

In [22]:
## vdb call

In [23]:

vdb=Chroma(persist_directory="./chroma_db_lorem3",
        embedding_function=embedding_model)

In [24]:
retriever=vdb.as_retriever(search_kwargs={"k": 35})

In [25]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder(
    "BAAI/bge-reranker-v2-m3"
)

Loading weights: 100%|██████████| 393/393 [00:00<00:00, 5896.36it/s]


In [26]:
def rerank_docs(question, docs, top_k=15):

    pairs = [
        (question, doc.page_content)
        for doc in docs
    ]

    scores = reranker.predict(pairs)

    scored_docs = list(zip(docs, scores))

    scored_docs.sort(
        key=lambda x: x[1],
        reverse=True
    )

    reranked_docs = [
        doc for doc, score in scored_docs[:top_k]
    ]

    return reranked_docs

In [34]:
llm = ChatGroq(
    model= "llama-3.3-70b-versatile",#llama-3.3-70b-versatile",
    temperature=0
)

In [29]:
prompt = ChatPromptTemplate.from_template("""
Tu es un assistant expert en analyse de documents administratifs français.

Tu dois répondre UNIQUEMENT à partir du contexte fourni.

Règles importantes :
- Ne jamais inventer d'information.
- Si une information n'est pas présente dans le contexte, répondre exactement :
  "Information non trouvée dans les documents."
- Lorsque l'information est trouvée :
  - citer le numéro de page,
  - citer la section si disponible,
  - résumer clairement la réponse.
- Si plusieurs pages parlent du sujet :
  - synthétiser les informations,
  - citer toutes les pages pertinentes.

Contexte :
{context}

Question :
{question}

Réponse :
""")

In [30]:
chain = prompt | llm | StrOutputParser()

In [31]:
def format_docs(docs):

    formatted = []

    for doc in docs:

        formatted.append(f"""
Source: {doc.metadata.get("source")}
Page: {doc.metadata.get("page")}
Section: {doc.metadata.get("header")}

{doc.page_content}
""")

    return "\n\n".join(formatted)

In [35]:
# without re ranker
question = "comment calculer le besoin ?"

docs = retriever.invoke(question)

context = format_docs(docs)

response = chain.invoke({
    "context": context,
    "question": question
})

print(response)

**Calcul du besoin (besoin brut → besoin net)**  

1. **Calcul du besoin brut**  
   - Formule :  
     \[
     \text{Besoin brut}= \frac{\text{Consommation de la période explorée}}{\text{nb jours de la période explorée}} \times \text{nb jours de consommation (selon le type de délai)}
     \]  
   - Cette règle apparaît dans la section **« Si Seuil de réappro > Disponible théorique »** (p. 15) 【Page 15 – Section Si Seuil de réappro > Disponible théorique】.  

2. **Calcul du seuil de réappro** (première étape de l’algorithme)  
   - \[
     \text{Seuil}= \frac{\text{Consommation de la période explorée}}{\text{nb jours de la période explorée}} \times \text{nb jours (selon le type de délai)}
     \]  
   - Décrit dans la section **« L’algorithme de calcul est le suivant »** (p. 14) 【Page 14 – Section L’algorithme de calcul est le suivant】.  

3. **Calcul du disponible théorique**  
   - Formule :  
     \[
     \text{Disponible théorique}= \text{Stock physique} - \text{Alloué} - \text{Man

In [36]:
# with re ranker
question = "comment calculer le besoin ?"

docs = retriever.invoke(question)
docs = rerank_docs(question, docs)
context = format_docs(docs)

response = chain.invoke({
    "context": context,
    "question": question
})

print(response)

**Calcul du besoin selon le guide « Calcul des besoins »**

1. **Déterminer le type de délai de l’article**  
   - Le champ « Type de délai » peut prendre les valeurs : blanc, 1 (D1), 2 (D2), SRA, SRM, STAT 1 (voir page None, section *Le champ « Type de délai » peut prendre 6 valeurs*).  
   - Selon le type choisi, le système utilise les paramètres :  
     - **Blanc** → XNBJRCBN1 / XNBJRCONS1 (jours sans délai)  
     - **1 / D1** → XNBJRCBN2 / XNBJRCONS2 (jours délai 1)  
     - **2 / D2** → XNBJRCBN3 / XNBJRCONS3 (jours délai 2)  
     - **SRM** → aucun calcul de consommation ; le besoin = seuil de réappro manuel renseigné sur la fiche article/site (page None, section *Le champ « Type de délai »*).

2. **Calcul du besoin brut**  
   - Formule :  
     \[
     \text{Besoin brut} = \frac{\text{Consommation de la période explorée}}{\text{Nb jours de la période explorée}} \times \text{Nb jours de consommation pour le calcul du besoin}
     \]  
   - Le « Nb jours de consommation » dépen

In [38]:
# with re ranker
question = " est-il possible d’avoir une visualisation multi-agents ?"

docs = retriever.invoke(question)
docs = rerank_docs(question, docs)
context = format_docs(docs)

response = chain.invoke({
    "context": context,
    "question": question
})

print(response)

Information non trouvée dans les documents.
